# SynthID Text prototype v3: current Transformers only

This notebook is designed for the **working Colab runtime you already have**.

It deliberately avoids:

- the legacy `synthid-text` package;
- JAX;
- old pinned Torch versions;
- old pinned Transformers versions;
- torchvision.

Instead, it uses only the SynthID primitives built into the current Hugging Face Transformers installation.

The detector is a small logistic classifier trained on **real keyed SynthID `g`-value features**. That gives us a practical detector without reproducing the old Bayesian training stack.

This is **not a Claude detector**. We generate and verify our own watermark.

The eventual browser version can use the same ingredients:

1. tokenize edited text;
2. compute keyed `g`-values;
3. aggregate them into a small feature vector;
4. apply a tiny linear detector;
5. update the evidence live.


## 0. Do not reinstall your environment

You have already got a Colab runtime in which current Transformers and PyTorch import correctly.

**Do not run another broad `pip install -U ...` command.**

This notebook starts by checking that the required SynthID classes are present.


In [ ]:
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    SynthIDTextWatermarkingConfig,
    SynthIDTextWatermarkLogitsProcessor,
)

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Watermark configuration

We use the standard example key sequence shown in the Transformers SynthID documentation.

The key is visible here because this is an educational demo. In a provider deployment, the verification configuration would normally be controlled.

The same configuration is used for generation and detection.


In [ ]:
MODEL_NAME = "openai-community/gpt2"

WATERMARK_KEYS = [654, 400, 836, 123, 340, 443, 597, 160, 57]
NGRAM_LEN = 5
SAMPLING_TABLE_SEED = 0
SAMPLING_TABLE_SIZE = 65536
CONTEXT_HISTORY_SIZE = 1024

watermark_config = SynthIDTextWatermarkingConfig(
    keys=WATERMARK_KEYS,
    ngram_len=NGRAM_LEN,
    sampling_table_seed=SAMPLING_TABLE_SEED,
    sampling_table_size=SAMPLING_TABLE_SIZE,
    context_history_size=CONTEXT_HISTORY_SIZE,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Model loaded.")


## 2. Generation helpers

The detector should score only the generated continuation, not the prompt.

We use a small collection of prompts rather than one repeated prompt so that the detector learns the watermark signal rather than quirks of a single topic.


In [ ]:
PROMPTS = [
    "Scientific models are useful even when they simplify reality because",
    "A good explanation can be simple without being misleading because",
    "One reason visualisations help people understand uncertainty is that",
    "The difference between evidence and certainty matters because",
    "When a model has several plausible ways to continue a sentence,",
    "A statistical pattern can be invisible to a reader while still being detectable because",
    "The most useful scientific abstractions often leave details out because",
    "An interactive explanation can reveal something that static prose cannot because",
]

GENERATION_KWARGS = dict(
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    max_new_tokens=180,
)

def generate_continuation(prompt, watermarked, seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]

    kwargs = dict(GENERATION_KWARGS)
    if watermarked:
        kwargs["watermarking_config"] = watermark_config

    with torch.no_grad():
        output = model.generate(**inputs, **kwargs)

    continuation_ids = output[:, prompt_len:].detach().cpu()
    continuation_text = tokenizer.decode(
        continuation_ids[0],
        skip_special_tokens=True,
    ).strip()

    return continuation_text


## 3. Recover keyed SynthID features from arbitrary text

The current Transformers implementation exposes the same logits processor used during generation.

Given a token sequence, it can recompute the keyed `g`-values implied by that sequence.

For each watermark depth we calculate the mean `g`-value over usable positions. The result is a tiny feature vector with one value per key.

Unlike v1, we **do not collapse these dimensions by hand**. We let a detector learn which combination of depths is actually informative.


In [ ]:
def make_logits_processor():
    return SynthIDTextWatermarkLogitsProcessor(
        ngram_len=NGRAM_LEN,
        keys=WATERMARK_KEYS,
        sampling_table_size=SAMPLING_TABLE_SIZE,
        sampling_table_seed=SAMPLING_TABLE_SEED,
        context_history_size=CONTEXT_HISTORY_SIZE,
        device=torch.device("cpu"),
        skip_first_ngram_calls=False,
        debug_mode=False,
    )


def synthid_features(text):
    ids = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].cpu()

    token_count = int(ids.shape[1])

    if token_count < NGRAM_LEN:
        return None

    processor = make_logits_processor()

    with torch.no_grad():
        g_values = processor.compute_g_values(ids).float()
        repetition_mask = processor.compute_context_repetition_mask(ids).bool()

        # EOS masking is not needed here because skip_special_tokens=True
        # removes the generated EOS from the text before rescoring.
        valid_mask = repetition_mask

    usable = int(valid_mask.sum().item())

    if usable == 0:
        return None

    # g_values shape: [batch, positions, watermark_depth]
    # valid_mask shape: [batch, positions]
    valid_g = g_values[valid_mask]

    # One mean value per watermark depth.
    means = valid_g.mean(dim=0).cpu().numpy().astype(float)

    return {
        "features": means,
        "tokens": token_count,
        "usable_positions": usable,
    }


## 4. Build a detector training set

We generate ordinary and watermarked continuations using the same model and generation settings.

For the first run, `N_PER_CLASS = 40` is enough to see whether the feature space separates.

If this works, we can later increase it for better calibration.


In [ ]:
N_PER_CLASS = 40

rows = []

for label, is_watermarked in [(0, False), (1, True)]:
    for i in range(N_PER_CLASS):
        prompt = PROMPTS[i % len(PROMPTS)]
        seed = 1000 + i + (10000 if is_watermarked else 0)

        text = generate_continuation(
            prompt=prompt,
            watermarked=is_watermarked,
            seed=seed,
        )

        result = synthid_features(text)

        if result is None:
            continue

        rows.append({
            "label": label,
            "type": "watermarked" if label == 1 else "ordinary",
            "seed": seed,
            "prompt": prompt,
            "text": text,
            "tokens": result["tokens"],
            "usable_positions": result["usable_positions"],
            **{
                f"g_mean_{j}": float(v)
                for j, v in enumerate(result["features"])
            },
        })

dataset_df = pd.DataFrame(rows)

print(dataset_df["type"].value_counts())
dataset_df.head()


## 5. Inspect the watermark-depth features

Before fitting anything, look at the average feature values for ordinary and watermarked text.

If there is a watermark signal, at least some depths should shift systematically.


In [ ]:
feature_cols = [f"g_mean_{j}" for j in range(len(WATERMARK_KEYS))]

feature_summary = dataset_df.groupby("type")[feature_cols].agg(["mean", "std"])
feature_summary


In [ ]:
mean_by_type = dataset_df.groupby("type")[feature_cols].mean().T
mean_by_type.index = [f"depth {i}" for i in range(len(feature_cols))]

fig, ax = plt.subplots(figsize=(9, 4.5))

x = np.arange(len(feature_cols))
width = 0.36

ax.bar(
    x - width / 2,
    mean_by_type["ordinary"],
    width,
    label="ordinary",
)

ax.bar(
    x + width / 2,
    mean_by_type["watermarked"],
    width,
    label="watermarked",
)

ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in range(len(feature_cols))])
ax.set_xlabel("Watermark depth")
ax.set_ylabel("Mean keyed g-value")
ax.set_title("Keyed SynthID features by watermark depth")
ax.legend()

plt.show()


## 6. Train a tiny logistic detector

We use a plain linear logistic model:

\[
p = \sigma(w^T x + b)
\]

The model has only a handful of parameters.

That is intentional. If this works, the final JavaScript implementation only needs a vector of weights and a bias.

We use a deterministic train/test split and standardise the features using the training set.


In [ ]:
rng = np.random.default_rng(2026)

indices = np.arange(len(dataset_df))
rng.shuffle(indices)

split = int(len(indices) * 0.75)

train_idx = indices[:split]
test_idx = indices[split:]

X = dataset_df[feature_cols].to_numpy(dtype=np.float32)
y = dataset_df["label"].to_numpy(dtype=np.float32)

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]

feature_mean = X_train.mean(axis=0)
feature_sd = X_train.std(axis=0)

# Avoid division by zero in pathological cases.
feature_sd = np.where(feature_sd == 0, 1.0, feature_sd)

X_train_z = (X_train - feature_mean) / feature_sd
X_test_z = (X_test - feature_mean) / feature_sd

Xt = torch.tensor(X_train_z, dtype=torch.float32)
yt = torch.tensor(y_train[:, None], dtype=torch.float32)

detector = torch.nn.Linear(Xt.shape[1], 1)
optimizer = torch.optim.Adam(detector.parameters(), lr=0.03)
loss_fn = torch.nn.BCEWithLogitsLoss()

for step in range(2000):
    optimizer.zero_grad()
    logits = detector(Xt)
    loss = loss_fn(logits, yt)
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(step, float(loss))

with torch.no_grad():
    weights = detector.weight.detach().cpu().numpy()[0]
    bias = float(detector.bias.detach().cpu().numpy()[0])

print("Weights:", weights)
print("Bias:", bias)


## 7. Evaluate on held-out samples

The output called `detector_score` is a classifier score between 0 and 1.

At this stage it should be read as **our demo detector's score**, not as a universal probability that text is AI-generated.

What matters first is whether held-out ordinary and watermarked samples separate.


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def detector_scores_from_matrix(X_raw):
    Xz = (X_raw - feature_mean) / feature_sd
    logits = Xz @ weights + bias
    return sigmoid(logits)


test_scores = detector_scores_from_matrix(X_test)

test_results = pd.DataFrame({
    "label": y_test.astype(int),
    "type": np.where(y_test == 1, "watermarked", "ordinary"),
    "detector_score": test_scores,
})

test_results.groupby("type")["detector_score"].agg(
    ["mean", "std", "min", "max"]
)


In [ ]:
pred = (test_scores >= 0.5).astype(int)
accuracy = (pred == y_test.astype(int)).mean()

print("Held-out accuracy at 0.5 threshold:", accuracy)

fig, ax = plt.subplots(figsize=(8, 4.5))

for label, group in test_results.groupby("type"):
    ax.hist(
        group["detector_score"],
        bins=np.linspace(0, 1, 11),
        alpha=0.55,
        label=label,
    )

ax.set_xlabel("Demo detector score")
ax.set_ylabel("Count")
ax.set_title("Held-out ordinary vs watermarked text")
ax.legend()

plt.show()


### Stop here if the distributions do not separate

This is our checkpoint.

If ordinary and watermarked held-out scores still overlap badly, do not proceed to the interactive work.

Send me:

- the histogram;
- the grouped score table;
- the feature-depth chart.

If they separate reasonably, continue.


## 8. Score arbitrary text

This function is now very close to what the browser will eventually do.

It:

1. tokenizes the current text;
2. recomputes keyed SynthID features;
3. standardises them;
4. applies the learned linear detector.


In [ ]:
def score_text(text):
    result = synthid_features(text)

    if result is None:
        return {
            "detector_score": float("nan"),
            "tokens": 0,
            "usable_positions": 0,
        }

    x = result["features"].astype(np.float32)
    xz = (x - feature_mean) / feature_sd
    logit = float(xz @ weights + bias)
    score = float(sigmoid(logit))

    return {
        "detector_score": score,
        "tokens": result["tokens"],
        "usable_positions": result["usable_positions"],
        "features": result["features"],
    }


## 9. Pick a strong watermarked sample for the editing experiment

We generate several candidates and select a sample that the held-out detector scores strongly.

This avoids building the essay around an unlucky low-signal passage.


In [ ]:
candidate_rows = []

for i in range(12):
    prompt = PROMPTS[i % len(PROMPTS)]
    text = generate_continuation(
        prompt=prompt,
        watermarked=True,
        seed=30000 + i,
    )
    score = score_text(text)

    candidate_rows.append({
        "i": i,
        "prompt": prompt,
        "text": text,
        "detector_score": score["detector_score"],
        "tokens": score["tokens"],
        "usable_positions": score["usable_positions"],
    })

candidates_df = pd.DataFrame(candidate_rows).sort_values(
    "detector_score",
    ascending=False,
)

candidates_df[
    ["i", "detector_score", "tokens", "usable_positions"]
].head()


In [ ]:
best = candidates_df.iloc[0]

watermarked_text = best["text"]

print("Chosen detector score:", best["detector_score"])
print()
print(watermarked_text)


## 10. Controlled editing experiment

For validation only, we replace increasing fractions of visible words and rescore the result.

This is not an automatic removal tool. It is a robustness experiment.

The final website will instead let the reader freely edit a textarea.


In [ ]:
def perturb_words(text, fraction, seed=42):
    rng = random.Random(seed)
    words = text.split()

    if not words:
        return text

    n_replace = min(len(words), round(len(words) * fraction))
    indexes = rng.sample(range(len(words)), n_replace)

    replacements = [
        "different", "simple", "useful", "clear", "general",
        "important", "possible", "common", "basic", "other",
    ]

    for idx in indexes:
        words[idx] = rng.choice(replacements)

    return " ".join(words)


edit_rows = []

for fraction in np.linspace(0, 0.5, 11):
    edited = perturb_words(
        watermarked_text,
        float(fraction),
        seed=42,
    )

    result = score_text(edited)

    edit_rows.append({
        "fraction_changed": float(fraction),
        "detector_score": result["detector_score"],
        "tokens": result["tokens"],
        "usable_positions": result["usable_positions"],
        "text": edited,
    })

edit_df = pd.DataFrame(edit_rows)

edit_df[
    [
        "fraction_changed",
        "detector_score",
        "tokens",
        "usable_positions",
    ]
]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(
    edit_df["fraction_changed"] * 100,
    edit_df["detector_score"],
    marker="o",
)

ax.set_ylim(0, 1)
ax.set_xlabel("Words replaced (%)")
ax.set_ylabel("Demo detector score")
ax.set_title("Watermark evidence as the passage is edited")

plt.show()


## 11. Passage-length experiment

The detector should generally become less informative as we remove usable text.

This will help us design the browser UI for cases where the reader deletes most of the passage.


In [ ]:
full_ids = tokenizer(
    watermarked_text,
    return_tensors="pt",
    add_special_tokens=False,
)["input_ids"][0]

length_rows = []

for n in [25, 40, 60, 80, 100, 120, 150, len(full_ids)]:
    n = min(int(n), int(len(full_ids)))

    truncated_text = tokenizer.decode(
        full_ids[:n],
        skip_special_tokens=True,
    )

    result = score_text(truncated_text)

    length_rows.append({
        "tokens_retained": n,
        "detector_score": result["detector_score"],
        "usable_positions": result["usable_positions"],
    })

length_df = pd.DataFrame(length_rows)
length_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(
    length_df["tokens_retained"],
    length_df["detector_score"],
    marker="o",
)

ax.set_ylim(0, 1)
ax.set_xlabel("Tokens retained")
ax.set_ylabel("Demo detector score")
ax.set_title("Watermark evidence vs passage length")

plt.show()


## 12. Manual edit playground

Edit the text in `edited_text`, then rerun the cell.

This is the direct notebook analogue of Version B.


In [ ]:
edited_text = watermarked_text

manual = score_text(edited_text)

print("Detector score:", manual["detector_score"])
print("Tokens:", manual["tokens"])
print("Usable watermark positions:", manual["usable_positions"])


## 13. Export everything needed for the Quarto prototype

The JSON contains the tiny detector itself:

- watermark configuration;
- feature means and standard deviations;
- linear detector weights;
- detector bias;
- one watermarked sample;
- editing and length trajectories.

The only substantial missing browser component after this is matching GPT-2 tokenization and the keyed `g`-value computation.


In [ ]:
fixture = {
    "implementation": "current Hugging Face Transformers SynthID primitives",
    "model_name": MODEL_NAME,
    "watermark": {
        "keys": WATERMARK_KEYS,
        "ngram_len": NGRAM_LEN,
        "sampling_table_seed": SAMPLING_TABLE_SEED,
        "sampling_table_size": SAMPLING_TABLE_SIZE,
        "context_history_size": CONTEXT_HISTORY_SIZE,
    },
    "detector": {
        "feature_mean": feature_mean.astype(float).tolist(),
        "feature_sd": feature_sd.astype(float).tolist(),
        "weights": weights.astype(float).tolist(),
        "bias": float(bias),
        "held_out_accuracy_at_0_5": float(accuracy),
    },
    "sample": {
        "text": watermarked_text,
        **{
            k: v
            for k, v in score_text(watermarked_text).items()
            if k != "features"
        },
    },
    "edit_trajectory": edit_df[
        [
            "fraction_changed",
            "detector_score",
            "tokens",
            "usable_positions",
        ]
    ].to_dict(orient="records"),
    "length_trajectory": length_df.to_dict(orient="records"),
}

output_path = Path("watermark-demo-fixture-v3.json")
output_path.write_text(
    json.dumps(fixture, indent=2),
    encoding="utf-8",
)

print("Wrote:", output_path.resolve())


## What to send me

The first checkpoint is now much simpler.

Please send:

1. the **watermark-depth feature chart**;
2. the **held-out ordinary vs watermarked histogram**;
3. the grouped score table printed above that histogram;
4. if those look good, the **editing curve**;
5. `watermark-demo-fixture-v3.json`.

If the held-out distributions do not separate, stop there. We will fix the detector before doing anything with Quarto.
